# Day 3: Data Quality, Preprocessing & Stationarity Analysis
**Prepared by:** Aditya (Lead), Anhum (Data), Tanveer (EDA & Verification)

This interactive notebook demonstrates:
1. End-to-end cleaning and forward-fill calendar alignment (`DataPreprocessor`).
2. Rigorous econometric stationarity testing via Augmented Dickey-Fuller (ADF) & KPSS.
3. The **Memory vs. Stationarity Dilemma** in quantitative finance.
4. Marcos López de Prado's **Fractional Differentiation** algorithm finding the optimal $d^*$.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocess import DataPreprocessor
from src.utils.stationarity import StationarityTester, FractionalDifferentiator
from src.utils.finance_math import calculate_log_returns

sns.set_theme(style="whitegrid")
print("Modules successfully loaded.")

## 1. Load Preprocessed Market Data
We load cleaned Apple (`AAPL`) and S&P 500 (`SPY`) data from `data/processed/`.

In [ ]:
aapl_df = pd.read_parquet("../data/processed/AAPL_processed.parquet")
price_aapl = aapl_df["Adj Close"]
ret_aapl = calculate_log_returns(price_aapl)

print(f"AAPL Daily Bars: {len(aapl_df)} from {aapl_df.index.min().date()} to {aapl_df.index.max().date()}")
aapl_df.tail()

## 2. Rolling Statistics: Raw Price vs. Log Returns
Observe how the raw price has an exploding, time-varying mean (non-stationary), while log returns fluctuate stably around zero with constant unconditional mean.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Raw Price
axes[0].plot(price_aapl, color="#1f77b4", label="AAPL Adj Close")
axes[0].plot(price_aapl.rolling(60).mean(), color="black", ls="--", label="60-Day Rolling Mean")
axes[0].set_title("Non-Stationary Raw Price Level (Exploding Mean & Trend)", fontsize=12, fontweight="bold")
axes[0].legend(loc="upper left")

# Log Returns
axes[1].plot(ret_aapl, color="#2ca02c", lw=0.8, alpha=0.8, label="Daily Log Returns")
axes[1].axhline(0, color="black", ls="--", alpha=0.7)
axes[1].set_title("Stationary Daily Log Returns (Mean Reverting around Zero)", fontsize=12, fontweight="bold")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

## 3. Joint Econometric Testing: ADF and KPSS

In [ ]:
diag_price = StationarityTester.diagnose_stationarity(price_aapl, "AAPL Raw Price")
diag_ret = StationarityTester.diagnose_stationarity(ret_aapl, "AAPL Log Returns")

pd.DataFrame([diag_price, diag_ret])

## 4. The Memory vs. Stationarity Dilemma & Fractional Differentiation
Standard integer differentiation ($d=1$) removes all price memory. Using fractional differentiation, we evaluate $d \in [0, 1]$ to find the minimal $d^*$ that achieves stationarity while retaining over $80\%$ correlation with historical price levels.

In [ ]:
min_d, sweep_df = FractionalDifferentiator.find_min_d_for_stationarity(price_aapl)

fig, ax1 = plt.subplots(figsize=(10, 5))

color = "#1f77b4"
ax1.set_xlabel("Differentiation Order (d)", fontsize=11)
ax1.set_ylabel("ADF Test Statistic", color=color, fontsize=11)
ax1.plot(sweep_df["d"], sweep_df["adf_stat"], color=color, marker="o", lw=2, label="ADF Statistic")
ax1.axhline(-2.86, color="red", ls="--", label="95% Critical Value (-2.86)")
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
color = "#2ca02c"
ax2.set_ylabel("Correlation with Original Price (Memory)", color=color, fontsize=11)
ax2.plot(sweep_df["d"], sweep_df["correlation_memory"], color=color, marker="s", lw=2, ls="-.", label="Memory Correlation")
ax2.tick_params(axis="y", labelcolor=color)

plt.title(f"AAPL: Memory vs. Stationarity Trade-Off (Optimal d* = {min_d:.2f})", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()